# Audio（音频处理）
Audio（音频处理），它本身就可以算是一个模态（单模态），但同时也是多模态系统的一个重要部分（比如「看图听音」）。
## 1. 音频处理常见任务
| 任务类型	| 输入| 	输出| 	典型模型 / 库 |
| --- | --- | --- | --- |
| 自动语音识别 (ASR) |	语音|	文本|	Whisper、Wav2Vec2、HuBERT|
|语音合成 (TTS)|	文本|	语音|	Tacotron 2、VITS、FastSpeech|
|音频分类 / 声音事件检测|	音频|	标签（动物叫声、警报声等）|	AudioSpectrogramTransformer、AST|
|说话人识别 / 验证|	语音|	说话人 ID 或相似度|	ECAPA-TDNN、SpeakerNet|
|音乐相关任务|	音频|	类型分析、节奏识别、分轨等|	MusicGen、Demucs|

## 2. 自动语音识别（ASR）
Automatic Speech Recognition
> 安装必备的基础环境（注意，重启内核）
> - `apt install ffmpeg`
> - `pip install ffmpeg-python`

In [8]:
from transformers import pipeline
from IPython.display import Audio

# 
# 如果要识别中文或多语言语音，推荐使用多语言版本 openai/whisper-large-v3
# whisper-tiny：最小、速度快、准确率低
# whisper-base：轻量级
# whisper-small：中等
# whisper-medium：更准
# whisper-large-v3：最准确、需更多显存
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")

audio_file = "audio/audio_2.wav"
Audio(audio_file)

result = asr(audio_file)
print("检测语言并识别文本：", result["text"])

Device set to use cuda:0


检测语言并识别文本： 原因是有人愿为该村电资四八万元费用修桥


### 3. 文本转语音（TTS）
> 安装可能所需的依赖: `pip install sentencepiece soundfile`
> - SentencePiece 是一个文本预处理和分词库，主要用于将文本转换为模型可以理解的格式。
> - SoundFile 是一个音频文件读写库，基于 libsndfile 构建。


In [11]:
from transformers import pipeline
from IPython.display import Audio


tts = pipeline("text-to-speech", "suno/bark")

speech = tts("SentencePiece 是一个文本预处理和分词库，主要用于将文本转换为模型可以理解的格式", forward_params={"do_sample": True})
print(speech)

Audio(speech['audio'], rate=speech['sampling_rate'])



Device set to use cuda:0
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.


{'audio': array([[ 4.9988425e-04, -4.6303641e-05,  1.4981799e-04, ...,
         2.8743234e-05,  3.4246681e-05,  3.7669073e-05]], dtype=float32), 'sampling_rate': 24000}


### 4. 文本转音频（TTA）
> 根据文本提示生成音乐

In [2]:

from transformers import pipeline
import scipy

import torch
import gc

# 清空 CUDA 缓存
torch.cuda.empty_cache()

# 强制垃圾回收
gc.collect()

synthesiser = pipeline("text-to-audio", "facebook/musicgen-large")

music = synthesiser("lo-fi music with a soothing melody", forward_params={"do_sample": True})

# 保存
scipy.io.wavfile.write("musicgen_out.wav", rate=music["sampling_rate"], data=music["audio"])

from IPython.display import Audio
Audio("musicgen_out.wav")

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.40it/s]
Device set to use cuda:0
